Groundwater | Transport Modeling Track

# Step 2: Perceptual Model – From Flow to Transport

> **The 10 steps at a glance:** 1-Goal → **2-Perceptual** → 3-MODFLOW → 4-Build → 5-Calibrate → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**Story so far:** In Step 1, you set the objective for the transport model: track a **conservative tracer** moving between the injection and abstraction wells of a **groundwater heat-exchange (GWHE) doublet** — paired wells used for heating and cooling, studied here as a solute-transport problem. The operational question is **recirculation / short-circuiting**: does the water (and any conservative signature it carries) injected at the reinjection well travel back to the paired abstraction well, and how fast? Now we build a **perceptual model for transport** – what additional understanding do we need beyond the calibrated flow model?

## Timing and Learning Objectives

| | |
|---|---|
| **Time** | ~30–35 min core + ~10 min optional sections |
| **Prerequisites** | Flow track Steps 1–2, Transport track Step 1 |

**Learning objectives:**

By the end of this notebook you will be able to:

1. Calculate **groundwater (seepage) velocity** from Darcy flux and effective porosity
2. Estimate **hydrodynamic dispersion** and set the longitudinal/transverse dispersivities $\alpha_L,\ \alpha_T$
3. Assess the **grid Peclet number** and the TVD advection scheme for the Limmat Valley
4. Explain how a doublet **injection source** is represented (well flux × concentration)
5. Identify the main **simplifications** in the perceptual model — including the transverse-smearing limitation

> **You will MODIFY a provided model, not build one.** The GWT (groundwater transport) model is supplied in Step 4; here we establish the parameters and concepts behind it.

In [ ]:
# Setup
import sys
import os

import pyproj
os.environ['PROJ_LIB'] = pyproj.datadir.get_data_dir()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

sys.path.append('../../_SUPPORT/src')
sys.path.append('../../_SUPPORT/src/scripts/scripts_exercises')

from data_utils import download_named_file, get_default_data_folder
from map_utils import display_concessions_map
from shared_functions import check_task_with_solution, create_multiple_choice

DATA_DIR = get_default_data_folder()

---
## 1 — From Flow to Transport

The calibrated flow model gives us the **velocity field** (specific discharge $q$). To simulate transport of a conservative tracer we need additional information:

| Flow model gave us | Transport adds |
|---|---|
| Hydraulic conductivity $K$ | Effective porosity $n_e$ → **velocity** $v = q / n_e$ |
| Hydraulic gradient $i$ | Dispersivity $\alpha_L,\ \alpha_T$ → **hydrodynamic dispersion** |
| Well pumping / injection rates | Source concentration $c_{inj}$ → **mass loading at the injection well** |
| Water balance (fluxes) | Tracer concentration of each flux → **transport boundary conditions** |

This notebook works through each of these additions.

---
## 2 — Groundwater Velocity

The **specific discharge** (Darcy flux) $q = Ki$ tells us the volume of water crossing a unit area per unit time, but it is not the speed at which water (or a tracer) actually moves through the pore space.

The **seepage velocity** (average linear velocity) is:

$$v = \frac{q}{n_e} = \frac{Ki}{n_e}$$

For the Limmat Valley glaciofluvial gravels, effective porosity $n_e$ is typically **0.15–0.30** (Freeze & Cherry 1979, Table 2.4). We adopt $n_e = 0.20$ as a reasonable central estimate for well-sorted gravel.

> **Predict before you run:** Near the doublet's abstraction well, pumping steepens the local hydraulic gradient. Before running the next cell, predict: will near-well seepage velocities be **higher or lower** than the domain-average velocity? Note your reasoning, then check it against the printed domain average.

In [ ]:
# Representative values from the calibrated flow model
K = 2.31e-3        # hydraulic conductivity [m/s]  (≈ 200 m/d, calibrated value)
dh = 18.0          # head difference across model domain [m]
dx = 7900.0        # domain extent in flow direction [m]
i = dh / dx        # hydraulic gradient [-]
n_e = 0.20         # effective porosity [-]

# Specific discharge
q = K * i  # [m/s]

# Seepage velocity
v = q / n_e  # [m/s]
v_m_per_day = v * 86400  # convert to m/day

print(f"Hydraulic gradient:     i = {i:.4f}")
print(f"Specific discharge:     q = {q:.2e} m/s  ({q * 86400:.2f} m/d)")
print(f"Seepage velocity:       v = {v:.2e} m/s  ({v_m_per_day:.1f} m/d)")

# Travel time across ~4.5 km domain
L_domain = 4500  # [m]
travel_time_days = L_domain / v_m_per_day
print(f"\nTravel time across {L_domain/1000:.1f} km domain: ~{travel_time_days:.0f} days ({travel_time_days/365:.0f} years)")
print(f"\nNote: this is a domain-average velocity. Near a doublet's abstraction")
print(f"well, pumping-induced gradients create local velocities well above it.")

**Key insight:** The seepage velocity (~2.3 m/day on average) is faster than the specific discharge (~0.45 m/day) by a factor of $1/n_e = 5$. This distinction matters for:

- **Wellhead / capture-zone protection** — defined by travel time of water (velocity), not flux
- **Tracer arrival times** — near a pumping well, pumping-induced gradients mean an injected tracer a few hundred metres upgradient could arrive within weeks to months
- **Doublet recirculation** — whether injected return-water short-circuits back to the paired abstraction well depends directly on these local velocities

### Checkpoint 1 — Seepage Velocity

In [ ]:
check_task_with_solution('task_t02_checkpoint_1')

---
## 3 — Dispersion

In a porous medium, a solute front does not advance as a sharp plug — it **spreads** due to:

1. **Mechanical dispersion** — velocity variations at the pore scale cause differential advection
2. **Molecular diffusion** — random motion of dissolved molecules (usually negligible in fast-flowing gravels)

The **longitudinal hydrodynamic dispersion coefficient** combines both:

$$D_L = \alpha_L \cdot v + D_m^*$$

where $\alpha_L$ is the longitudinal dispersivity [m] and $D_m^*$ is the effective molecular diffusion coefficient. In porous media $D_m^* \approx 10^{-9}$ m$^2$/s, which in the model's time units (days) is $D_m^* \approx 8.64\times10^{-5}$ m$^2$/d (the $\times 86400$ s/d conversion — never use the unconverted $10^{-9}$ in a days-based model).

### Scale-dependent dispersivity

Dispersivity is notoriously **scale-dependent**: it increases with the distance over which transport is observed (Gelhar et al. 1992). For our model scale (~1–5 km):

| Observation scale | Typical $\alpha_L$ range | Reference |
|---|---|---|
| Laboratory column (0.1–1 m) | 0.001–0.01 m | Freeze & Cherry 1979 |
| Field tracer test (10–100 m) | 0.1–10 m | Gelhar et al. 1992 |
| Regional model (1–10 km) | 5–100 m | Gelhar et al. 1992 |

For the Limmat Valley at our model scale we adopt **$\alpha_L = 10$ m**. The transverse dispersivity is taken as one-tenth of the longitudinal value, **$\alpha_T = \alpha_L / 10 = 1$ m** — a tracer spreads roughly ten times less across the flow direction than along it.

> **Forward pointer:** $\alpha_T = 1$ m is small. In the grid-Peclet check below you will see that resolving *transverse* spreading is far harder than longitudinal spreading — the **transverse Peclet number stays well above the stability target even on a refined grid**. Step 4 returns to this as the *transverse-smearing* limitation.

In [ ]:
# Dispersion parameters
alpha_L = 10.0          # longitudinal dispersivity [m]
alpha_T = alpha_L / 10  # transverse dispersivity [m]  → 1 m
D_m_star = 1e-9         # effective molecular diffusion [m²/s]  (= 8.64e-5 m²/d in model days-units)

D_L = alpha_L * v + D_m_star  # longitudinal dispersion coefficient [m²/s]
D_T = alpha_T * v + D_m_star  # transverse  dispersion coefficient [m²/s]

print(f"Longitudinal dispersivity:  alpha_L = {alpha_L} m")
print(f"Transverse dispersivity:    alpha_T = {alpha_T} m  (= alpha_L / 10)")
print(f"Dispersion coefficient:     D_L = {D_L:.2e} m²/s")
print(f"  of which mechanical:      alpha_L * v = {alpha_L * v:.2e} m²/s")
print(f"  of which diffusion:       D_m* = {D_m_star:.0e} m²/s  (= 8.64e-5 m²/d)")
print(f"\n→ Mechanical dispersion dominates over molecular diffusion by a")
print(f"  factor of {alpha_L * v / D_m_star:.0e}. Molecular diffusion is negligible.")

# --- Grid Peclet number: numerical resolution criterion (Pe_grid ≲ 2) ---
# Note: v is in m/s and dx in m, so Pe is dimensionless regardless of time units;
#       cp1 reports v in m/d, here we keep SI (m/s) — the Peclet ratio is unaffected.
dx_grid = 49  # typical cell size in the provided DISV grid [m] (model median)
Pe_L = v * dx_grid / D_L   # longitudinal grid Peclet
Pe_T = v * dx_grid / D_T   # transverse  grid Peclet

print(f"\nNative grid (Δx = {dx_grid} m):")
print(f"  Longitudinal grid Peclet:  Pe_L = {Pe_L:.1f}   (target ≤ 2)")
print(f"  Transverse grid Peclet:    Pe_T = {Pe_T:.0f}   (target ≤ 2 — badly under-resolved)")
print(f"  Critical cell size for Pe_L ≤ 2:  Δx ≤ {2 * D_L / v:.0f} m = 2 × alpha_L")
print(f"  Critical cell size for Pe_T ≤ 2:  Δx ≤ {2 * D_T / v:.0f} m = 2 × alpha_T")

Our **longitudinal** grid Peclet number is ~5, exceeding the classical limit of 2. With a **central-difference** advection scheme this would cause spurious oscillations. MODFLOW 6 GWT supports **TVD (Total Variation Diminishing)** advection, which stays stable at higher grid Peclet numbers, so the longitudinal front is usable on the native grid (at the cost of some numerical dispersion).

The **transverse** grid Peclet number is far worse (~49 on the native grid, and still ~10 even on the refined grid used in Step 4). TVD keeps the solution stable, but it cannot recover detail the grid is too coarse to carry: lateral spreading is **systematically smeared**. We flag this now and return to it as a judgment limitation in Step 4 — it is *taught*, not *fixed*.

<details>
<summary><b>Optional: Grid Peclet criterion — when does it matter?</b> (click to expand)</summary>

The **grid Peclet number** is a numerical (not physical) criterion:

$$Pe_{grid} = \frac{v \cdot \Delta x}{D} \lesssim 2$$

Longitudinally this simplifies to $\Delta x \lesssim 2\,\alpha_L$ (= 20 m for $\alpha_L = 10$ m); transversely to $\Delta x \lesssim 2\,\alpha_T$ (= 2 m for $\alpha_T = 1$ m). Resolving transverse spreading therefore demands a ~10× finer grid than longitudinal spreading — rarely affordable over a km-scale domain.

**Why we rely on TVD instead of refining everywhere:**
- TVD (upstream-weighted) schemes are stable regardless of $Pe_{grid}$ — they handle sharp fronts without oscillation
- The tradeoff is **numerical dispersion**, which adds to the physical dispersion
- Refining the entire grid to a few metres would multiply cell count and runtime prohibitively

**When local refinement IS used:** around the doublet, where concentrations are highest and gradients steepest, local refinement (Step 4) reduces $Pe_L$ to ~1 — but $Pe_T$ still stays near 10, which is exactly the transverse-smearing limitation above.

</details>

### Checkpoint — Grid Peclet Number

In [ ]:
check_task_with_solution('task_t02_checkpoint_pe')

---
### Optional — reactive solutes (sorption)

The **core path of this course is a conservative tracer** ($R = 1$): it moves with the water and is neither retained nor degraded. Many real solutes are *not* conservative — they sorb onto grain surfaces and travel slower than the water. This is optional background; the doublet study uses a conservative tracer.

<details>
<summary><b>Retardation by sorption (click to expand)</b></summary>

For a linearly-sorbing solute at equilibrium, the **retardation factor** is:

$$R_s = 1 + \frac{\rho_b \, K_d}{n_e}$$

where $\rho_b$ is the dry bulk density [kg/m³], $K_d$ is the distribution coefficient [m³/kg], and $n_e$ is the effective porosity. The sorbing solute's front advances at $v / R_s$.

**Worked example** (the values used in the checkpoint below): for $\rho_b \approx 1900$ kg/m³, $K_d \approx 5\times10^{-4}$ m³/kg, and $n_e = 0.20$:

$$R_s = 1 + \frac{1900 \times 5\times10^{-4}}{0.20} = 1 + \frac{0.95}{0.20} \approx 5.75$$

so the solute front travels about **5.75× slower** than the conservative tracer.

| | Conservative tracer | Sorbing solute |
|---|---|---|
| **Retardation $R$** | 1 | $> 1$ (here ≈ 5.75) |
| **Front velocity** | $v$ | $v / R_s$ |
| **Predictability** | High | Low — $K_d$ depends on species, pH, organic content |
| **Examples** | chloride, bromide, many dyes | many heavy metals and organics |

In MODFLOW 6 GWT this is handled by the **MST** package (`sorption`, `bulk_density`, `distcoef`); first-order decay adds `decay`. We keep these **off** for the conservative core path.

</details>

**Checkpoint — compute the sorption retardation factor** for the worked-example values above (answer in the cell that follows):

In [ ]:
check_task_with_solution('task_t02_checkpoint_2')

---
## 4 — GWHE Doublet Locations in the Limmat Valley

The Limmat Valley is one of Switzerland's most intensely used aquifers for **groundwater heat exchange (GWHE)** — **injection–abstraction doublet wells** that pump groundwater, exchange heat for heating or cooling, and reinject it nearby. Although their purpose is thermal, each doublet is also a **hydraulic** feature: it withdraws water at one well and returns it at another, setting up exactly the injection→abstraction flow path along which a conservative tracer could recirculate.

In the regulatory data these appear as groundwater heat-pump (WPG = *Wärmepumpe Grundwasser*) and cooling-water (KW = *Kühlwasser*) concessions. Let's map them in our model area.

In [ ]:
# Download well and boundary data
wells_path = download_named_file(name='wells', data_type='gis')
model_boundary_path = download_named_file(name='model_boundary', data_type='gis')

# Read and clip to model boundary
gdf_boundary = gpd.read_file(model_boundary_path)
wells_gdf = gpd.read_file(wells_path, layer='GS_GRUNDWASSERFASSUNGEN_OGD_P')
wells_gdf = wells_gdf.clip(gdf_boundary)

# Add concession ID
wells_gdf['concession_id'] = wells_gdf['GWR_ID'].str.split('_').str[0]

# Filter active wells
is_decommissioned = wells_gdf['BESCHREIBUNG'].str.contains('aufgehoben', case=False, na=False)
is_unused = wells_gdf['BESCHREIBUNG'].str.contains('ungenutzt', case=False, na=False)
wells_active = wells_gdf[~is_decommissioned & ~is_unused].copy()

# Filter for GWHE doublet use: WPG (heat pump) or KW (cooling water)
is_gwhe = wells_active['NUTZART'].str.contains('WPG|KW', case=False, na=False)
gwhe_wells = wells_active[is_gwhe].copy()

n_gwhe_concessions = gwhe_wells['concession_id'].nunique()
n_all_concessions = wells_active['concession_id'].nunique()

print(f"Active concessions in model area:  {n_all_concessions}")
print(f"GWHE doublet concessions (WPG/KW): {n_gwhe_concessions}")
print(f"Fraction GWHE:                     {n_gwhe_concessions/n_all_concessions:.0%}")

In [ ]:
# Map of GWHE doublet concessions
display_concessions_map(
    gwhe_wells,
    boundary_gdf=gdf_boundary,
    map_title="GWHE Doublet Concessions (WPG/KW) — Limmat Valley"
)

**Observations from the map:**

- GWHE doublets are **concentrated in the city centre** (Zurich districts 1–5), reflecting high heating/cooling demand
- Individual doublets are not represented in our base model — students add a specific injection→abstraction pair in the application step
- Each doublet is a candidate **recirculation / short-circuiting** setting: a conservative tracer injected at the reinjection well may travel back to the paired abstraction well, and that path runs through exactly the velocity and dispersion field we just characterised

### Representing the injection source (preview)

How does the model put tracer *into* the aquifer at the injection well? In Step 4 we will use the **SSM (Source–Sink Mixing)** package with an **auxiliary concentration** on the injection well. The well already supplies a water flux $Q_{inj}$ (from the flow model), and SSM multiplies it by a specified concentration $c_{inj}$ to give a **mass loading rate**

$$\dot{m} = Q_{inj} \cdot c_{inj} \quad [\text{M/T}].$$

Because the mass rate is set by the *flux* (not by cell geometry), it is **grid-independent** — refining the grid does not change how much tracer enters.

> **Contrast — why not a fixed-concentration cell?** A **CNC (constant-concentration)** cell is a *Dirichlet* boundary: it pins $C = c_{inj}$ at the cell and lets the model supply whatever mass flux is needed to hold that value. That mass then depends on the local flow field and grid resolution — **the model decides the loading, not you**. For a controlled, grid-independent injection we therefore use **WEL flux + SSM concentration**, not CNC. (Full build: Step 4.)

### Checkpoint — GWHE Doublet Distribution

In [ ]:
create_multiple_choice('task_t02_checkpoint_3')

---
## 5 — Could We Be Wrong?

Every perceptual model involves simplifications. Here are the main **conceptual alternatives** for our conservative-tracer transport model:

| Assumption | Alternative | Impact if wrong |
|---|---|---|
| Conservative tracer ($R = 1$, no decay) | Sorbing / decaying solute | Slower, attenuated arrival; $R_s$ must be estimated per species |
| Uniform $\alpha_L = 10$ m | Larger/smaller or spatially variable $\alpha_L$ | More/less front spreading; peak concentration and arrival shift |
| $\alpha_T = \alpha_L/10 = 1$ m, **adequately resolved** | Transverse spreading **under-resolved on any feasible grid** ($Pe_T \approx 49$ native, ~10 refined) | **Lateral plume width is numerically smeared — threshold *area* claims are not defensible** |
| Single vertical layer | Multiple layers to resolve vertical structure | Vertical concentration gradients missed |
| Steady-state flow field | Transient flow (seasonal water-table changes) | Seasonal changes in path and travel time missed |
| Uniform porosity $n_e = 0.20$ | Spatially variable $n_e$ (0.15–0.30) | Local velocity variations, different travel times |
| TVD advection adequate on native grid | Numerical dispersion biases the front | Front artificially smoothed; local refinement (Step 4) needed near the doublet |

The flagship limitation is the **transverse-smearing** one: even after local refinement the transverse grid Peclet number stays well above 2, so the model's **lateral** plume width is dominated by numerical, not physical, dispersion. This is *taught*, not *fixed* — it directly shapes what statements we can defend in the application step (receptor arrival and well concentration: yes; contaminated *area* contours: no). Step 4 makes this concrete.

*Before moving on, take a moment to reflect:*

> **Which assumption above do you think is most consequential for the doublet recirculation question? Why?**
>
> Consider both (a) how likely the assumption is to be wrong and (b) how much impact it would have on the predicted tracer arrival at the abstraction well.

---
## What You're Taking Forward

| Parameter | Value | Unit | Source |
|---|---|---|---|
| Effective porosity $n_e$ | 0.20 | — | Literature (glaciofluvial gravel) |
| Seepage velocity $v$ | ~2.3 (domain average) | m/day | $v = Ki/n_e$ from calibrated model |
| Longitudinal dispersivity $\alpha_L$ | 10 | m | Gelhar et al. 1992 (model scale) |
| Transverse dispersivity $\alpha_T$ | 1 | m | $\alpha_T = \alpha_L/10$ |
| Molecular diffusion $D_m^*$ | $8.64\times10^{-5}$ | m²/d | $=10^{-9}$ m²/s; negligible vs. mechanical dispersion |
| Native longitudinal grid Peclet $Pe_L$ | ~5 | — | $v\,\Delta x / D_L$ (> 2 → use TVD) |
| Native transverse grid Peclet $Pe_T$ | ~49 | — | $v\,\Delta x / D_T$ (under-resolved; ~10 even refined) |
| Advection scheme | TVD | — | Stable for $Pe_{grid} > 2$ |
| Injection source | WEL flux $\times$ SSM concentration | — | mass $= Q_{inj}\,c_{inj}$ (grid-independent) |
| Optional sorption retardation $R_s$ | $1 + \rho_b K_d/n_e$ | — | Off for conservative core path |

> **Next step:** In [Step 3 (MODFLOW for Transport)](./03t_modflow_transport.ipynb), we translate this perceptual model into a MODFLOW 6 **GWT (Groundwater Transport)** model configuration.

---
## References

- Doppler, T. et al. (2007). Field evidence of a dynamic leakage coefficient for modelling river–aquifer interactions. *Journal of Hydrology*, 347(1–2), 177–187. [doi:10.1016/j.jhydrol.2007.09.017](https://doi.org/10.1016/j.jhydrol.2007.09.017)
- Fetter, C.W. (2001). *Applied Hydrogeology*. 4th ed. Prentice Hall. [ISBN 978-0131226876](https://www.pearson.com/en-us/subject-catalog/p/applied-hydrogeology/P200000006370)
- Freeze, R.A. & Cherry, J.A. (1979). *Groundwater*. Prentice Hall. [ISBN 978-0133653120](https://www.pearson.com/en-us/subject-catalog/p/groundwater/P200000006244)
- Gelhar, L.W., Welty, C. & Rehfeldt, K.R. (1992). A critical review of data on field-scale dispersion in aquifers. *Water Resources Research*, 28(7), 1955–1974. [doi:10.1029/92WR00607](https://doi.org/10.1029/92WR00607)
- Zheng, C. & Bennett, G.D. (2002). *Applied Contaminant Transport Modeling*. 2nd ed. Wiley. [ISBN 978-0471384779](https://www.wiley.com/en-us/Applied+Contaminant+Transport+Modeling%2C+2nd+Edition-p-9780471384779)
- Canton of Zurich (2023). *Gewässer- und Grundwasser-Jahrbuch*. AWEL. [zh.ch](https://www.zh.ch/de/umwelt-tiere/wasser-gewaesser/gewaesserschutz/gewaesserqualitaet.html)